# RFM Customer Segmentation Analysis

This notebook converts raw online retail transaction data into customer-level RFM segments. The goal is to identify valuable customers, repeat-purchase customers, reactivation opportunities, and product or country patterns that can support targeted marketing decisions.

RFM stands for:

- **Recency:** how recently a customer purchased.
- **Frequency:** how often a customer purchased.
- **Monetary value:** how much revenue a customer generated.


## 1. Import Libraries and Load Data

The notebook is designed to run from the repository root or from inside the `notebooks/` folder. It looks for the raw dataset in the standard project data folder first, then falls back to a local notebook path.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATHS = [
    Path("../data/data.csv"),
    Path("data/data.csv"),
    Path("data.csv"),
]

DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Save the raw online retail dataset as data/data.csv before running this notebook."
    )

df = pd.read_csv(DATA_PATH, encoding="latin-1")
df.head()

## 2. Data Understanding

Before calculating RFM metrics, the dataset is inspected for shape, columns, missing values, and basic numeric distributions.


In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.info()

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "Column", 0: "Missing Values"})
    .query("`Missing Values` > 0")
    .sort_values("Missing Values", ascending=False)
)

missing_summary

## 3. Data Cleaning

For valid customer-level RFM analysis, transactions must have a known customer, valid invoice date, positive quantity, and positive unit price.

The Online Retail dataset can include cancelled orders. These are commonly identified by invoice numbers starting with `C`. Cancelled transactions are excluded from the main RFM analysis because RFM is intended to measure completed customer purchasing behavior.


In [ ]:
clean_df = df.copy()

# Standardize column names expected in the Online Retail dataset.
expected_columns = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
}

missing_columns = expected_columns.difference(clean_df.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype(str)
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"], errors="coerce")
clean_df["Description"] = clean_df["Description"].fillna("Not Provided")

clean_df["IsCancelled"] = clean_df["InvoiceNo"].str.startswith("C", na=False)

cleaning_summary = {
    "raw_rows": len(clean_df),
    "cancelled_rows": int(clean_df["IsCancelled"].sum()),
    "missing_customer_rows": int(clean_df["CustomerID"].isna().sum()),
    "invalid_date_rows": int(clean_df["InvoiceDate"].isna().sum()),
    "non_positive_quantity_rows": int((clean_df["Quantity"] <= 0).sum()),
    "non_positive_price_rows": int((clean_df["UnitPrice"] <= 0).sum()),
}

cleaning_summary

In [ ]:
clean_df = clean_df[
    (~clean_df["IsCancelled"])
    & clean_df["CustomerID"].notna()
    & clean_df["InvoiceDate"].notna()
    & (clean_df["Quantity"] > 0)
    & (clean_df["UnitPrice"] > 0)
].copy()

clean_df["CustomerID"] = clean_df["CustomerID"].astype(int)
clean_df["TotalAmount"] = clean_df["Quantity"] * clean_df["UnitPrice"]

print(f"Cleaned transaction rows: {len(clean_df):,}")
clean_df.head()

## 4. RFM Metric Calculation

RFM metrics are calculated at the customer level:

- **Recency:** number of days since the customer's most recent purchase.
- **Frequency:** number of unique invoices placed by the customer.
- **Monetary:** total purchase value generated by the customer.

The analysis date is set one day after the latest invoice date in the dataset, which keeps the calculation reproducible.


In [ ]:
analysis_date = clean_df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    clean_df.groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda x: (analysis_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalAmount", "sum"),
        Country=("Country", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
    )
    .reset_index()
)

rfm.head()

## 5. RFM Scoring

Quartile-based scoring converts raw RFM metrics into comparable 1–4 scores.

- Lower recency is better, so the most recent customers receive higher recency scores.
- Higher frequency is better.
- Higher monetary value is better.

Rank-based quartiles are used to reduce errors caused by repeated values.


In [ ]:
rfm["RecencyScore"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    q=4,
    labels=[4, 3, 2, 1],
).astype(int)

rfm["FrequencyScore"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4],
).astype(int)

rfm["MonetaryScore"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4],
).astype(int)

rfm["RFMScore"] = (
    rfm["RecencyScore"].astype(str)
    + rfm["FrequencyScore"].astype(str)
    + rfm["MonetaryScore"].astype(str)
)

rfm.head()

## 6. Customer Segmentation Logic

Customers are assigned to practical business segments based on their RFM score pattern.

The segment rules are intentionally business-friendly rather than overly complex:

- **High-Value Customers:** recent, frequent, and high-spending customers.
- **Potential Loyal Customers:** customers with strong recent activity and repeat-purchase potential.
- **Big Spenders:** customers with high monetary value, even if their frequency is not always the highest.
- **Recent Customers:** customers who purchased recently but need nurturing into repeat buyers.
- **Churn Risk Customers:** customers with low recency and weak engagement signals.
- **Other Customers:** customers who do not strongly match the above groups.


In [ ]:
def assign_segment(row: pd.Series) -> str:
    recency = row["RecencyScore"]
    frequency = row["FrequencyScore"]
    monetary = row["MonetaryScore"]

    if recency >= 4 and frequency >= 4 and monetary >= 4:
        return "High-Value Customers"

    if recency >= 3 and frequency >= 3:
        return "Potential Loyal Customers"

    if monetary >= 4 and frequency >= 2:
        return "Big Spenders"

    if recency >= 3:
        return "Recent Customers"

    if recency <= 2 and frequency <= 2 and monetary <= 2:
        return "Churn Risk Customers"

    return "Other Customers"


rfm["Segment"] = rfm.apply(assign_segment, axis=1)
rfm.head()

## 7. Segment Summary

This table summarizes customer count, average recency, average frequency, average monetary value, and total revenue by segment. It is the main business output of the analysis.


In [ ]:
segment_summary = (
    rfm.groupby("Segment")
    .agg(
        Customers=("CustomerID", "count"),
        AverageRecency=("Recency", "mean"),
        AverageFrequency=("Frequency", "mean"),
        AverageMonetary=("Monetary", "mean"),
        TotalRevenue=("Monetary", "sum"),
    )
    .sort_values("TotalRevenue", ascending=False)
    .round(2)
)

segment_summary

## 8. Visual Analysis

The following charts compare customer segments by count, monetary value, frequency, and recency. Bar charts are used for comparing numeric metrics across segments because they are clearer than pie charts for average values.


In [ ]:
segment_order = segment_summary.index.tolist()

plt.figure(figsize=(12, 6))
sns.countplot(data=rfm, y="Segment", order=segment_order)
plt.title("Customer Count by Segment")
plt.xlabel("Number of Customers")
plt.ylabel("Segment")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=rfm,
    y="Segment",
    x="Monetary",
    order=segment_order,
    estimator=np.mean,
    errorbar=None,
)
plt.title("Average Monetary Value by Segment")
plt.xlabel("Average Monetary Value")
plt.ylabel("Segment")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=rfm,
    y="Segment",
    x="Recency",
    order=segment_order,
    estimator=np.mean,
    errorbar=None,
)
plt.title("Average Recency by Segment")
plt.xlabel("Average Recency in Days")
plt.ylabel("Segment")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=rfm,
    y="Segment",
    x="Frequency",
    order=segment_order,
    estimator=np.mean,
    errorbar=None,
)
plt.title("Average Purchase Frequency by Segment")
plt.xlabel("Average Number of Orders")
plt.ylabel("Segment")
plt.tight_layout()
plt.show()

## 9. Country and Product-Level Patterns

Country and product analysis helps connect customer segments to practical marketing and merchandising decisions.


In [ ]:
country_summary = (
    rfm.groupby("Country")
    .agg(
        Customers=("CustomerID", "count"),
        TotalRevenue=("Monetary", "sum"),
        AverageMonetary=("Monetary", "mean"),
    )
    .sort_values("TotalRevenue", ascending=False)
    .round(2)
    .head(10)
)

country_summary

In [ ]:
top_products = (
    clean_df.groupby("Description")
    .agg(
        QuantitySold=("Quantity", "sum"),
        Revenue=("TotalAmount", "sum"),
    )
    .sort_values("Revenue", ascending=False)
    .round(2)
    .head(10)
)

top_products

## 10. Business Recommendations

The recommendations below translate the segmentation output into marketing actions.


In [ ]:
recommendations = pd.DataFrame(
    [
        {
            "Segment": "High-Value Customers",
            "Recommended Action": "Protect this group with loyalty rewards, early access offers, and personalized account-level communication.",
        },
        {
            "Segment": "Potential Loyal Customers",
            "Recommended Action": "Encourage repeat purchases through bundles, product recommendations, and loyalty program invitations.",
        },
        {
            "Segment": "Big Spenders",
            "Recommended Action": "Use premium offers, high-value product recommendations, and VIP-style promotions.",
        },
        {
            "Segment": "Recent Customers",
            "Recommended Action": "Send second-purchase campaigns, onboarding offers, and product education messages.",
        },
        {
            "Segment": "Churn Risk Customers",
            "Recommended Action": "Use reactivation campaigns, limited-time discounts, and feedback requests.",
        },
        {
            "Segment": "Other Customers",
            "Recommended Action": "Monitor behavior over time and use broad nurturing campaigns until clearer patterns emerge.",
        },
    ]
)

recommendations

## 11. Export Cleaned RFM Output

The cleaned customer-level segmentation output is exported for Power BI and reporting.


In [ ]:
output_paths = [
    Path("../data/cleaned_rfm_customer_segments.csv"),
    Path("data/cleaned_rfm_customer_segments.csv"),
]

output_path = output_paths[0] if output_paths[0].parent.exists() else output_paths[1]
output_path.parent.mkdir(parents=True, exist_ok=True)

rfm.to_csv(output_path, index=False)
print(f"Saved cleaned RFM output to: {output_path}")

## 12. Business Summary

This analysis follows a complete customer analytics workflow:

1. Cleaning transaction-level retail data.
2. Excluding invalid or cancelled transactions from customer-level analysis.
3. Creating Recency, Frequency, and Monetary metrics.
4. Scoring customers using rank-based quartiles.
5. Translating scores into business-friendly customer segments.
6. Producing segment summaries, visualizations, and recommendations for marketing action.
